# Large Area / Landing Area Model

This notebook trains a CNN-based binary classifier for possible helicopter landing/open areas:

- `LargeAreas`
- `NotLargerAreas` or `NotLargeAreas`

It expects your zip file to be in Google Drive at:

```text
/content/drive/MyDrive/NatDisaster/LargeAreas.zip
```

The trained model will be saved back to Google Drive as a `.pth` file.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Import Libraries

In [ ]:
import os
import zipfile
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ConvNeXt_Tiny_Weights, ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

## Settings

`convnext_tiny` is the default. It is a modern CNN and usually works very well for aerial image classification.

If Colab runs out of memory, lower `BATCH_SIZE` to `16`, or switch `MODEL_NAME` to `resnet50`.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ZIP_PATH = Path('/content/drive/MyDrive/NatDisaster/LargeAreas.zip')

# Optional fallback if you manually made a /drive shortcut.
if not ZIP_PATH.exists():
    fallback = Path('/drive/NatDisaster/LargeAreas.zip')
    if fallback.exists():
        ZIP_PATH = fallback

EXTRACT_DIR = Path('/content/LargeAreasData')
MODELS_DIR = Path('/content/drive/MyDrive/NatDisaster/models_largeareas')
RESULTS_DIR = Path('/content/drive/MyDrive/NatDisaster/results_largeareas')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'convnext_tiny'  # options: 'convnext_tiny', 'resnet50'
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
print('Zip path:', ZIP_PATH)

## Unzip Data

The zip should contain class folders like:

```text
LargeAreas/
NotLargerAreas/
```

This notebook also accepts `NotLargeAreas` if you fix the folder name later.

In [ ]:
assert ZIP_PATH.exists(), f'Could not find {ZIP_PATH}. Check the path and file name.'

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

print('Extracted to:', EXTRACT_DIR)
print('Top-level items:')
for p in sorted(EXTRACT_DIR.iterdir()):
    print('-', p.name)

## Clean Dataset Layout

This cell finds the class folders automatically, skips hidden Mac metadata files, verifies images, and copies everything into this clean layout:

```text
/content/LargeAreasImageFolder/
  LargeAreas/
  NotLargeAreas/
```

Even if your original folder is named `NotLargerAreas`, the clean output label will be `NotLargeAreas`.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

CLASS_ALIASES = {
    'LargeAreas': ['LargeAreas', 'Large_Areas', 'LargeArea', 'largeareas', 'landing', 'openarea'],
    'NotLargeAreas': ['NotLargerAreas', 'NotLargeAreas', 'Not_Larger_Areas', 'Not_Large_Areas', 'NotLargeArea', 'notlargeareas'],
}

def is_mac_metadata(path):
    parts = path.parts
    return (
        '__MACOSX' in parts
        or path.name.startswith('._')
        or path.name.startswith('.DS_Store')
        or any(part.startswith('._') for part in parts)
    )

def looks_like_image_path(path):
    return (
        path.is_file()
        and path.suffix.lower() in IMAGE_EXTS
        and not is_mac_metadata(path)
        and not path.name.startswith('.')
    )

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

def valid_image_files(folder):
    files = []
    for p in folder.rglob('*'):
        if looks_like_image_path(p) and is_valid_image(p):
            files.append(p)
    return sorted(files)

def count_images(folder):
    return len(valid_image_files(folder)) if folder else 0

def find_folder_for_aliases(root, aliases):
    alias_lower = {a.lower() for a in aliases}
    candidates = []
    for d in root.rglob('*'):
        if not d.is_dir():
            continue
        if '__MACOSX' in d.parts or d.name.startswith('._'):
            continue
        if d.name.lower() in alias_lower:
            n = count_images(d)
            if n > 0:
                candidates.append((n, str(d), d))
    if not candidates:
        return None
    return sorted(candidates, reverse=True)[0][2]

found = {}
for clean_name, aliases in CLASS_ALIASES.items():
    folder = find_folder_for_aliases(EXTRACT_DIR, aliases)
    found[clean_name] = folder
    print(clean_name, '->', folder, 'valid images:', count_images(folder) if folder else 0)

assert found['LargeAreas'] is not None, 'Could not find a valid LargeAreas folder in the zip.'
assert found['NotLargeAreas'] is not None, 'Could not find a valid NotLargerAreas/NotLargeAreas folder in the zip.'

CLEAN_DATA_DIR = Path('/content/LargeAreasImageFolder')
if CLEAN_DATA_DIR.exists():
    shutil.rmtree(CLEAN_DATA_DIR)
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

for clean_name, source_folder in found.items():
    target_folder = CLEAN_DATA_DIR / clean_name
    target_folder.mkdir(parents=True, exist_ok=True)
    files = valid_image_files(source_folder)
    skipped = 0
    for i, src in enumerate(files, start=1):
        dst = target_folder / f'{i:05d}.jpg'
        try:
            with Image.open(src) as img:
                img = ImageOps.exif_transpose(img).convert('RGB')
                img.save(dst, format='JPEG', quality=92)
        except Exception:
            skipped += 1
    print(f'{clean_name}: copied {len(files) - skipped}, skipped {skipped}')

print('\nClean dataset:')
for d in sorted(CLEAN_DATA_DIR.iterdir()):
    print(d.name, count_images(d))

## Preview Images

In [ ]:
def show_samples(data_dir, samples_per_class=6):
    class_dirs = sorted([p for p in data_dir.iterdir() if p.is_dir()])
    fig, axes = plt.subplots(len(class_dirs), samples_per_class, figsize=(samples_per_class * 2.2, len(class_dirs) * 2.2))
    if len(class_dirs) == 1:
        axes = np.array([axes])

    for row, class_dir in enumerate(class_dirs):
        files = [p for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
        chosen = random.sample(files, min(samples_per_class, len(files)))
        for col in range(samples_per_class):
            ax = axes[row, col]
            ax.axis('off')
            if col < len(chosen):
                img = Image.open(chosen[col]).convert('RGB')
                ax.imshow(img)
                ax.set_title(class_dir.name, fontsize=9)
    plt.tight_layout()
    plt.show()

show_samples(CLEAN_DATA_DIR)

## Create Train / Validation / Test Split

This uses a `70 / 15 / 15` split.

In [ ]:
base_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

full_dataset_for_split = datasets.ImageFolder(CLEAN_DATA_DIR, transform=base_transform)
class_names = full_dataset_for_split.classes
class_to_idx = full_dataset_for_split.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

targets = np.array(full_dataset_for_split.targets)
indices = np.arange(len(full_dataset_for_split))

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=targets,
    random_state=SEED,
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=targets[temp_idx],
    random_state=SEED,
)

print('Classes:', class_names)
print('Class to idx:', class_to_idx)
print('Train:', len(train_idx))
print('Val:', len(val_idx))
print('Test:', len(test_idx))

for split_name, split_idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    counts = pd.Series(targets[split_idx]).map(idx_to_class).value_counts()
    print('\n' + split_name)
    print(counts)

## Data Augmentation and DataLoaders

In [ ]:
if MODEL_NAME == 'convnext_tiny':
    weights = ConvNeXt_Tiny_Weights.DEFAULT
    normalizer = weights.transforms()
elif MODEL_NAME == 'resnet50':
    weights = ResNet50_Weights.DEFAULT
    normalizer = weights.transforms()
else:
    raise ValueError(f'Unknown MODEL_NAME: {MODEL_NAME}')

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalizer.mean, std=normalizer.std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalizer.mean, std=normalizer.std),
])

train_dataset_full = datasets.ImageFolder(CLEAN_DATA_DIR, transform=train_transform)
eval_dataset_full = datasets.ImageFolder(CLEAN_DATA_DIR, transform=eval_transform)

train_dataset = Subset(train_dataset_full, train_idx)
val_dataset = Subset(eval_dataset_full, val_idx)
test_dataset = Subset(eval_dataset_full, test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('DataLoaders ready')

## Build CNN Model

This uses a pretrained CNN backbone and replaces the final layer with a 2-class classifier.

In [ ]:
def build_model(model_name, num_classes):
    if model_name == 'convnext_tiny':
        model = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
    elif model_name == 'resnet50':
        model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
    else:
        raise ValueError(f'Unknown model: {model_name}')
    return model

model = build_model(MODEL_NAME, num_classes=len(class_names)).to(DEVICE)
print(model.__class__.__name__)

## Loss, Optimizer, and Class Weights

In [ ]:
train_targets = targets[train_idx]
class_counts = np.bincount(train_targets, minlength=len(class_names))
class_weights = len(train_targets) / (len(class_names) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print('Class counts:', {idx_to_class[i]: int(c) for i, c in enumerate(class_counts)})
print('Class weights:', {idx_to_class[i]: float(w) for i, w in enumerate(class_weights.detach().cpu())})

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

## Training Helpers

In [ ]:
def run_one_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                logits = model(images)
                loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        total_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_probs.extend(probs.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average='binary' if len(class_names) == 2 else 'macro',
        zero_division=0,
    )

    try:
        if len(class_names) == 2:
            positive_idx = class_to_idx.get('LargeAreas', 1)
            auc = roc_auc_score(all_labels, np.array(all_probs)[:, positive_idx])
        else:
            auc = roc_auc_score(all_labels, np.array(all_probs), multi_class='ovr')
    except Exception:
        auc = np.nan

    return {
        'loss': avg_loss,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': auc,
        'labels': np.array(all_labels),
        'preds': np.array(all_preds),
        'probs': np.array(all_probs),
    }

## Train Model

In [ ]:
best_val_f1 = -1
best_path = MODELS_DIR / f'{MODEL_NAME}_largeareas_best.pth'
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = run_one_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_one_epoch(model, val_loader, optimizer=None)
    scheduler.step()

    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_accuracy': train_metrics['accuracy'],
        'train_precision': train_metrics['precision'],
        'train_recall': train_metrics['recall'],
        'train_f1': train_metrics['f1'],
        'train_roc_auc': train_metrics['roc_auc'],
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1'],
        'val_roc_auc': val_metrics['roc_auc'],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train acc {row['train_accuracy']:.4f} f1 {row['train_f1']:.4f} loss {row['train_loss']:.4f} | "
        f"val acc {row['val_accuracy']:.4f} f1 {row['val_f1']:.4f} loss {row['val_loss']:.4f}"
    )

    if row['val_f1'] > best_val_f1:
        best_val_f1 = row['val_f1']
        checkpoint = {
            'model_name': MODEL_NAME,
            'state_dict': model.state_dict(),
            'image_size': IMAGE_SIZE,
            'class_names': list(class_names),
            'class_to_idx': dict(class_to_idx),
            'idx_to_class': {int(k): v for k, v in idx_to_class.items()},
            'best_val_f1': float(best_val_f1),
            'epoch': int(epoch),
        }
        torch.save(checkpoint, best_path)
        print('Saved best model to:', best_path)

history_df = pd.DataFrame(history)
history_csv = RESULTS_DIR / f'{MODEL_NAME}_largeareas_history.csv'
history_df.to_csv(history_csv, index=False)
print('Saved history to:', history_csv)
history_df

## Graph Training Curves

In [ ]:
def plot_metric(history_df, train_col, val_col, title, ylabel):
    plt.figure(figsize=(7, 4))
    plt.plot(history_df['epoch'], history_df[train_col], marker='o', label=train_col)
    plt.plot(history_df['epoch'], history_df[val_col], marker='o', label=val_col)
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

plot_metric(history_df, 'train_accuracy', 'val_accuracy', 'Accuracy by Epoch', 'Accuracy')
plot_metric(history_df, 'train_loss', 'val_loss', 'Loss by Epoch', 'Loss')
plot_metric(history_df, 'train_f1', 'val_f1', 'F1 Score by Epoch', 'F1')
plot_metric(history_df, 'train_roc_auc', 'val_roc_auc', 'ROC-AUC by Epoch', 'ROC-AUC')

## Evaluate on Test Set

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
model = build_model(checkpoint['model_name'], num_classes=len(checkpoint['class_names'])).to(DEVICE)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

class_names = checkpoint['class_names']
class_to_idx = checkpoint['class_to_idx']
idx_to_class = {int(k): v for k, v in checkpoint['idx_to_class'].items()}

test_metrics = run_one_epoch(model, test_loader, optimizer=None)

print('TEST METRICS')
print('Accuracy:', round(test_metrics['accuracy'], 4))
print('Precision:', round(test_metrics['precision'], 4))
print('Recall:', round(test_metrics['recall'], 4))
print('F1:', round(test_metrics['f1'], 4))
print('ROC-AUC:', round(test_metrics['roc_auc'], 4))
print()
print(classification_report(test_metrics['labels'], test_metrics['preds'], target_names=class_names, zero_division=0))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(test_metrics['labels'], test_metrics['preds'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap='Greens', values_format='d')
plt.title('Large Area Model Confusion Matrix')
plt.show()

cm_path = RESULTS_DIR / f'{MODEL_NAME}_largeareas_confusion_matrix.png'
fig.savefig(cm_path, dpi=200, bbox_inches='tight')
print('Saved confusion matrix to:', cm_path)

## Save Final `.pth` Model for the Website

This exports the best checkpoint with the information your app needs:

- model name
- weights
- class labels
- image size
- test metrics

In [ ]:
final_export_path = MODELS_DIR / f'{MODEL_NAME}_large_area_detector.pth'

safe_test_metrics = {
    'accuracy': float(test_metrics['accuracy']),
    'precision': float(test_metrics['precision']),
    'recall': float(test_metrics['recall']),
    'f1': float(test_metrics['f1']),
    'roc_auc': float(test_metrics['roc_auc']) if not np.isnan(test_metrics['roc_auc']) else None,
}

export_checkpoint = {
    'model_name': checkpoint['model_name'],
    'state_dict': checkpoint['state_dict'],
    'image_size': int(checkpoint['image_size']),
    'class_names': list(checkpoint['class_names']),
    'class_to_idx': dict(checkpoint['class_to_idx']),
    'idx_to_class': {int(k): v for k, v in checkpoint['idx_to_class'].items()},
    'test_metrics': safe_test_metrics,
}

torch.save(export_checkpoint, final_export_path)
print('Exported large area model to:', final_export_path)
print('Test metrics:', safe_test_metrics)

## Single Image Prediction Test

Use this later to test one aerial patch or sliding-window crop.

In [ ]:
def predict_large_area_image(image_path, model, checkpoint):
    image_path = Path(image_path)
    img = Image.open(image_path).convert('RGB')

    if checkpoint['model_name'] == 'convnext_tiny':
        weights = ConvNeXt_Tiny_Weights.DEFAULT
    elif checkpoint['model_name'] == 'resnet50':
        weights = ResNet50_Weights.DEFAULT
    else:
        raise ValueError(checkpoint['model_name'])

    normalizer = weights.transforms()
    transform = transforms.Compose([
        transforms.Resize((checkpoint['image_size'], checkpoint['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=normalizer.mean, std=normalizer.std),
    ])

    x = transform(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()

    class_names = checkpoint['class_names']
    result = {class_names[i]: float(probs[i]) for i in range(len(class_names))}
    predicted_class = class_names[int(np.argmax(probs))]
    return predicted_class, result

# Example usage:
# test_image = '/content/LargeAreasImageFolder/LargeAreas/00001.jpg'
# pred, probs = predict_large_area_image(test_image, model, checkpoint)
# print(pred, probs)